# RDT-1B Feature-level BlurIG
**What this does:** Runs ManiSkill simulation to get a real robot camera frame, then applies Blur Integrated Gradients in SigLIP patch-embedding space to show which parts of the image most influenced RDT's predicted gripper action.

**Before running:** Set `Runtime > Change runtime type > A100 GPU`

**Runtime:** ~5-10 min on A100 (first run downloads ~2GB of models)

In [ ]:
import os
if os.path.exists('/content/.deps_installed'):
    print('Already installed — skip to Cell 2.')
else:
    os.system('git clone https://github.com/thu-ml/RoboticsDiffusionTransformer')
    os.system('git clone https://github.com/rdgbrandon/rdt-igtesting')
    os.system('apt-get install -qq libvulkan1 libegl1-mesa-dev libgles2-mesa-dev libosmesa6-dev')
    os.system('pip install -q einops timm pyyaml mani-skill accelerate')
    os.system('pip install -q "numpy<2"')
    os.system('pip install -q "protobuf>=5.28.0"')
    open('/content/.deps_installed', 'w').close()
    print('Install done. Restarting runtime...')
    os.kill(os.getpid(), 9)

In [ ]:
import os, traceback, shutil
from huggingface_hub import hf_hub_download

os.environ['RDT_REPO']       = '/content/RoboticsDiffusionTransformer'
os.environ['LANG_EMBED']     = 'lang_embed.pt'
os.environ['MANISKILL_TASK'] = 'PickCube-v1'
os.environ['SKIP_IG']        = '1'
os.chdir('/content/rdt-igtesting')
!git pull -q

shutil.copy(
    hf_hub_download('robotics-diffusion-transformer/maniskill-model',
                    'lang_embeds/text_embed_PickCube-v1.pt'),
    'lang_embed.pt',
)

try:
    %run rdt_blurig.py
except Exception:
    print(traceback.format_exc())

In [ ]:
import gymnasium as gym, mani_skill.envs, types
import torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image as PILImage
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
import shutil

_TASKS = {
    "PickCube-v1":         ("pick up the cube",  "Grasp a red cube and move it to a target goal position."),
    "StackCube-v1":        ("stack the cubes",    "Pick up a red cube and stack it on top of a green cube and let go of the cube without it falling."),
    "PushCube-v1":         ("push the cube",      "Push and move a cube to a goal region in front of it."),
    "PegInsertionSide-v1": ("insert the peg",     "Pick up a orange-white peg and insert the orange end into the box with a hole in it."),
    "PlugCharger-v1":      ("plug the charger",   "Pick up one of the misplaced shapes on the board/kit and insert it into the correct empty slot."),
}
JOINT_NAMES = ["base rot", "shoulder", "upper arm", "elbow",
               "forearm rot", "wrist pitch", "wrist rot", "gripper"]

# ── Task input ────────────────────────────────────────────────────────────────
_input = input("Task (e.g. 'pick up the cube'): ").strip().lower()

def _match(q):
    words = set(q.split())
    return max(_TASKS, key=lambda t: len(words & set((_TASKS[t][0]+" "+_TASKS[t][1]).lower().split())))

task_id = _match(_input)
t_short, t_desc = _TASKS[task_id]
print(f"Matched: {task_id}\n  {t_desc}\n")

# ── Language embedding ────────────────────────────────────────────────────────
shutil.copy(
    hf_hub_download('robotics-diffusion-transformer/maniskill-model',
                    f'lang_embeds/text_embed_{task_id}.pt'),
    'lang_embed.pt',
)
_ld = torch.load('lang_embed.pt', map_location="cpu", weights_only=False)
_lt = _ld if not isinstance(_ld, dict) else _ld["embeddings"]
if _lt.dim() == 2: _lt = _lt.unsqueeze(0)
lang_tokens    = _lt.to(DEVICE, dtype=DTYPE)
lang_attn_mask = torch.ones(lang_tokens.shape[:2], dtype=torch.bool, device=DEVICE)
with torch.no_grad():
    lang_cond = rdt.lang_adaptor(lang_tokens)
L_emb = lang_tokens.shape[1]
print(f"Lang embed: {lang_tokens.shape}")

# ── Tokenize into words ───────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("t5-small")
token_ids = None
for _txt in [t_desc, t_short]:
    _ids = tokenizer(_txt, return_tensors="pt").input_ids[0]
    if len(_ids) == L_emb:
        token_ids = _ids; break

if token_ids is None:
    plot_labels = [f"t{i}" for i in range(L_emb)]
    group_map   = [[i] for i in range(L_emb)]
else:
    raw_toks = tokenizer.convert_ids_to_tokens(token_ids)
    groups, cur = [], [0]
    for i in range(1, len(raw_toks)):
        if raw_toks[i].startswith('▁'):
            groups.append(cur); cur = [i]
        else:
            cur.append(i)
    groups.append(cur)
    group_map   = groups
    plot_labels = [
        tokenizer.decode([token_ids[i].item() for i in g], skip_special_tokens=True).strip()
        or f"[{token_ids[g[0]].item()}]" for g in groups
    ]
    print(f"Words: {plot_labels}\n")
W = len(plot_labels)

# ── Initial frame for attribution ─────────────────────────────────────────────
_env0 = gym.make(task_id, obs_mode='rgb', render_mode='rgb_array',
                 control_mode='pd_joint_pos', sensor_configs={'width':384,'height':384})
_obs0, _ = _env0.reset(seed=42)
_cam0    = list(_obs0['sensor_data'].keys())[0]
_rgb0    = _obs0['sensor_data'][_cam0]['rgb']
if hasattr(_rgb0, 'cpu'): _rgb0 = _rgb0.cpu().numpy()
scene_frame = PILImage.fromarray(np.array(_rgb0).squeeze()).resize((384, 384))
_q0 = _obs0['agent']['qpos']
if hasattr(_q0, 'cpu'): _q0 = _q0.cpu().numpy()
_q0 = np.array(_q0).flatten()
_j8 = np.concatenate([_q0[:7], [float(_q0[7:9].mean())]])
_jn = (torch.tensor(_j8) - STATE_MIN) / (STATE_MAX - STATE_MIN).clamp(min=1e-6) * 2 - 1
_s0 = torch.zeros(128, dtype=DTYPE, device=DEVICE)
_s0[MANISKILL_INDICES] = _jn.to(DTYPE).to(DEVICE)
_prop0 = _s0.unsqueeze(0).unsqueeze(0)
_am0   = torch.zeros(1, 1, 128, dtype=DTYPE, device=DEVICE)
_am0[0, 0, MANISKILL_INDICES] = 1.0
with torch.no_grad():
    scene_emb   = encode_image(scene_frame)
    scene_state = rdt.state_adaptor(torch.cat([_prop0, _am0], dim=2))
_env0.close()

# ── Rollout ───────────────────────────────────────────────────────────────────
_envr = gym.make(task_id, obs_mode='rgb', render_mode='rgb_array',
                 control_mode='pd_joint_pos', sensor_configs={'width':384,'height':384})
_obsr, _ = _envr.reset(seed=42)
frames, step, done, _chunk, _cp = [], 0, False, None, 8

while not done and step < 80:
    _camr = list(_obsr['sensor_data'].keys())[0]
    _rgbr = _obsr['sensor_data'][_camr]['rgb']
    if hasattr(_rgbr, 'cpu'): _rgbr = _rgbr.cpu().numpy()
    frames.append(np.array(_rgbr).squeeze().copy())
    if _cp >= 8:
        _fn = PILImage.fromarray(frames[-1]).resize((384, 384))
        _qn = _obsr['agent']['qpos']
        if hasattr(_qn, 'cpu'): _qn = _qn.cpu().numpy()
        _qn = np.array(_qn).flatten()
        _jn2 = (torch.tensor(np.concatenate([_qn[:7],[float(_qn[7:9].mean())]]) - STATE_MIN.numpy()) /
                (STATE_MAX - STATE_MIN).clamp(min=1e-6).numpy() * 2 - 1)
        _sn  = torch.zeros(128, dtype=DTYPE, device=DEVICE)
        _sn[MANISKILL_INDICES] = torch.tensor(_jn2, dtype=DTYPE).to(DEVICE)
        _pn  = _sn.unsqueeze(0).unsqueeze(0)
        _an  = torch.zeros(1, 1, 128, dtype=DTYPE, device=DEVICE)
        _an[0, 0, MANISKILL_INDICES] = 1.0
        with torch.no_grad():
            _en  = encode_image(_fn)
            _icn = rdt.img_adaptor(_en.repeat(1, 6, 1))
            _stn = rdt.state_adaptor(torch.cat([_pn, _an], dim=2))
            _pr  = rdt.conditional_sample(lang_cond, lang_attn_mask, _icn, _stn, _an, ctrl_freqs)
        _chunk = ((_pr[0, :, MANISKILL_INDICES].cpu().float() + 1) / 2 *
                  (STATE_MAX - STATE_MIN) + STATE_MIN).numpy()
        _cp = 0
    _obsr, _, _term, _trunc, _info = _envr.step(_chunk[_cp].reshape(1, 8))
    done = _term or _trunc; _cp += 1; step += 1
_envr.close()
success = _info.get('success', '?')
print(f"Rollout: {step} steps  success={success}")

n_show = min(len(frames), 12)
idxs   = np.linspace(0, len(frames)-1, n_show, dtype=int)
fig_r, ax_r = plt.subplots(2, n_show//2, figsize=(3*(n_show//2), 6))
for idx, ax in zip(idxs, ax_r.flat):
    ax.imshow(frames[idx]); ax.set_title(f"step {idx}", fontsize=7); ax.axis("off")
fig_r.suptitle(f"{task_id} — {step} steps  success={success}", fontsize=10)
plt.tight_layout(); plt.savefig("rollout.png", dpi=120, bbox_inches="tight"); plt.show()

# ── Joint × Image (per-joint BlurIG) ──────────────────────────────────────────
n_j    = len(MANISKILL_INDICES)
_midx  = torch.tensor(MANISKILL_INDICES, device=DEVICE)
sigmas = torch.linspace(SIGMA_MAX, 0.0, N_BLURIG_STEPS + 1)
ji_tot = [torch.zeros_like(scene_emb) for _ in range(n_j)]

print(f"\nJoint × Image BlurIG ({N_BLURIG_STEPS} steps × {n_j} joints) ...")
for k in range(N_BLURIG_STEPS):
    E_t    = embed_blur(scene_emb.detach(), sigmas[k].item()).requires_grad_(True)
    E_next = embed_blur(scene_emb.detach(), sigmas[k+1].item())
    with torch.enable_grad():
        _ac = rdt.conditional_sample(lang_cond, lang_attn_mask,
                                     rdt.img_adaptor(E_t.repeat(1,6,1)),
                                     scene_state, _am0, ctrl_freqs)
    js = _ac[:, :SCORE_HORIZON, :][:, :, _midx].norm(dim=(0,1))
    for j in range(n_j):
        g = torch.autograd.grad(js[j], E_t, retain_graph=(j < n_j-1))[0]
        ji_tot[j] = ji_tot[j] + g.detach() * (E_next - E_t.detach())
    if (k+1) % 5 == 0: print(f"  step {k+1}/{N_BLURIG_STEPS}")

# ── Word × Joint (per-joint token IG) ─────────────────────────────────────────
N_IG   = 20
wj_tot = [torch.zeros_like(lang_tokens) for _ in range(n_j)]
_bl    = torch.zeros_like(lang_tokens)
_dl    = lang_tokens - _bl

print(f"\nWord × Joint token IG ({N_IG} steps × {n_j} joints) ...")
for k in range(N_IG):
    alpha  = (k + 0.5) / N_IG
    interp = (_bl + alpha * _dl).requires_grad_(True)
    with torch.enable_grad():
        _ac = rdt.conditional_sample(
            rdt.lang_adaptor(interp), lang_attn_mask,
            rdt.img_adaptor(scene_emb.detach().repeat(1,6,1)),
            scene_state, _am0, ctrl_freqs)
    js = _ac[:, :SCORE_HORIZON, :][:, :, _midx].norm(dim=(0,1))
    for j in range(n_j):
        g = torch.autograd.grad(js[j], interp, retain_graph=(j < n_j-1))[0]
        wj_tot[j] = wj_tot[j] + g.detach() * _dl.detach()
    if (k+1) % 5 == 0: print(f"  step {k+1}/{N_IG}")

wj_tot  = [t / N_IG for t in wj_tot]
attr_jt = np.array([wj_tot[j].squeeze(0).float().cpu().abs().sum(-1).numpy() for j in range(n_j)])
attr_jw = np.array([[attr_jt[j, g].sum() for g in group_map] for j in range(n_j)])  # (8, W)

# ── Word × Image ──────────────────────────────────────────────────────────────
# Option A: hook nn.MultiheadAttention and extract lang→image cross-attention
attn_data, _patched = [], []
try:
    def _make_hook(orig):
        def _fw(q, k, v, *a, **kw):
            kw['need_weights'] = True; kw['average_attn_weights'] = False
            out = orig(q, k, v, *a, **kw)
            if isinstance(out, tuple) and out[1] is not None:
                attn_data.append(out[1].detach().cpu())
            return out[0] if isinstance(out, tuple) else out
        return _fw

    for _nm, _m in rdt.named_modules():
        if isinstance(_m, torch.nn.MultiheadAttention):
            _patched.append((_m, _m.forward))
            _m.forward = _make_hook(_m.forward)

    with torch.no_grad():
        _ = rdt.conditional_sample(
            rdt.lang_adaptor(lang_tokens), lang_attn_mask,
            rdt.img_adaptor(scene_emb.detach().repeat(1,6,1)),
            scene_state, _am0, ctrl_freqs)
    print(f"\nOption A: {len(attn_data)} attn maps captured, "
          f"shapes: {sorted(set(tuple(a.shape) for a in attn_data))}")
finally:
    for _m, _orig in _patched:
        _m.forward = _orig

_n_img  = 6 * 729
use_attn = False
wi_maps  = None

if attn_data:
    _stacked = torch.cat(attn_data, dim=0)          # (total_maps, heads, N, N) or (maps, N, N)
    if _stacked.dim() == 3: _stacked = _stacked.unsqueeze(1)
    N_tok = _stacked.shape[-1]
    if N_tok >= L_emb + _n_img:
        _limg = _stacked[:, :, :L_emb, L_emb:L_emb+_n_img]   # (maps, heads, L, n_img)
        _avg  = _limg.mean(dim=(0,1)).numpy()                   # (L, n_img)
        _avg  = _avg.reshape(L_emb, 6, 729).mean(axis=1)       # (L, 729)
        _wi   = np.array([_avg[g, :].sum(axis=0) for g in group_map])  # (W, 729)
        _wi   = np.maximum(_wi, 0)
        wi_maps   = _wi.reshape(W, 27, 27)
        wi_maps   = wi_maps / (wi_maps.max(axis=(1,2), keepdims=True) + 1e-8)
        use_attn  = True
        print("Option A succeeded — using attention maps for Word × Image.")

if not use_attn:
    print("Option A: no matching layout. Using Option B (matrix product).")
    ji_flat  = np.array([np.abs(ji_tot[j].squeeze(0).float().cpu().numpy()).sum(-1) for j in range(n_j)])
    _jw_norm = attr_jw / (np.abs(attr_jw).max(axis=1, keepdims=True) + 1e-8)
    wi_flat  = np.maximum(_jw_norm.T @ ji_flat, 0)       # (W, 729)
    wi_maps  = wi_flat.reshape(W, 27, 27)
    wi_maps  = wi_maps / (wi_maps.max(axis=(1,2), keepdims=True) + 1e-8)

# ── Figures ───────────────────────────────────────────────────────────────────
img_np    = np.array(scene_frame) / 255.0
cmap      = plt.cm.inferno
_bil      = getattr(PILImage, "Resampling", PILImage).BILINEAR

def _upsample(m):
    return np.array(PILImage.fromarray((m*255).astype(np.uint8)).resize((384,384), _bil)) / 255.0

# Figure 1: Joint × Image
fig1, ax1 = plt.subplots(1, n_j+1, figsize=(4*(n_j+1), 4.5))
ax1[0].imshow(img_np); ax1[0].set_title("scene", fontsize=9); ax1[0].axis("off")
for j in range(n_j):
    amap = to_map(ji_tot[j], 384, 384)
    ax1[j+1].imshow(img_np); ax1[j+1].imshow(amap, cmap="inferno", alpha=0.6, vmin=0, vmax=1)
    ax1[j+1].set_title(JOINT_NAMES[j], fontsize=8); ax1[j+1].axis("off")
fig1.suptitle(f"Joint × Image | {task_id}", fontsize=10)
plt.tight_layout(); plt.savefig("attr_joint_image.png", dpi=130, bbox_inches="tight"); plt.show()

# Figure 2: Word × Joint
_jw_n = attr_jw / (attr_jw.max(axis=1, keepdims=True) + 1e-8)
fig2, (f2a, f2b) = plt.subplots(2, 1, figsize=(max(12,W*1.4), 8), gridspec_kw={"height_ratios":[3,1]})
im2 = f2a.imshow(_jw_n, cmap="inferno", aspect="auto", vmin=0, vmax=1)
f2a.set_xticks(range(W)); f2a.set_xticklabels(plot_labels, rotation=30, ha="right", fontsize=10)
f2a.set_yticks(range(n_j)); f2a.set_yticklabels(JOINT_NAMES, fontsize=10)
f2a.set_title(f"Word × Joint | {task_id}  (row-normalized)", fontsize=11)
plt.colorbar(im2, ax=f2a, fraction=0.02, pad=0.02)
_tot = attr_jw.sum(axis=0)
f2b.bar(range(W), _tot, color=[cmap(v) for v in _tot/(_tot.max()+1e-8)], edgecolor="white", lw=0.5)
f2b.set_xticks(range(W)); f2b.set_xticklabels(plot_labels, rotation=30, ha="right", fontsize=9)
f2b.set_ylabel("Total (all joints)")
fig2.suptitle(f"Word × Joint | {task_id}", fontsize=10, y=1.01)
plt.tight_layout(); plt.savefig("attr_word_joint.png", dpi=130, bbox_inches="tight"); plt.show()

# Figure 3: Word × Image
n_cols = min(W, 7); n_rows = (W + n_cols - 1) // n_cols
fig3, ax3 = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 3*n_rows), squeeze=False)
ax3f = ax3.flat
for w in range(W):
    ax3f[w].imshow(img_np)
    ax3f[w].imshow(_upsample(wi_maps[w]), cmap="inferno", alpha=0.65, vmin=0, vmax=1)
    ax3f[w].set_title(f'"{plot_labels[w]}"', fontsize=8); ax3f[w].axis("off")
for ax in list(ax3f)[W:]:
    ax.axis("off")
method = "attention" if use_attn else "matrix product"
fig3.suptitle(f"Word × Image ({method}) | {task_id}", fontsize=10)
plt.tight_layout(); plt.savefig("attr_word_image.png", dpi=130, bbox_inches="tight"); plt.show()

print("Saved: rollout.png  attr_joint_image.png  attr_word_joint.png  attr_word_image.png")

In [ ]:
from IPython.display import Image as IPyImage, display
for fname, title in [
    ("rollout.png",          "Rollout frames"),
    ("attr_joint_image.png", "Joint × Image"),
    ("attr_word_joint.png",  "Word × Joint"),
    ("attr_word_image.png",  "Word × Image"),
]:
    print(f"── {title} ──")
    display(IPyImage(fname))